# Práctica: Despliegue y uso de modelos en AI Foundry

Este repositorio contiene la práctica para aprender a desplegar y usar modelos en AI Foundry. La práctica está dividida en 3 partes independientes. Cada parte debe entregarse como un notebook Jupyter (`.ipynb`) con un nombre claro y apropiado.


## 1) Text, JSON y Guardrails
Objetivo: Desplegar un modelo y realizar tres tipos de interacciones:
### 1.1- Generar texto. 
Simple generación de texto con system prompt y user prompt.

⭐ Suma puntos crear un chat interactivo por CLI que persista la memoria a corto plazo.

[Documentación](https://learn.microsoft.com/en-us/azure/foundry/foundry-models/how-to/generate-responses?tabs=python)


In [3]:
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

project_endpoint = "https://pruebauno.services.ai.azure.com/api/projects/projuno"
# Build the base URL: project_endpoint + /openai/v1 (no api-version needed)
base_url = project_endpoint.rstrip("/") + "/openai/v1"

# get_bearer_token_provider returns a callable; call it to get automatic refresh of the token string
credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, "https://ai.azure.com/.default")
client = OpenAI(
    base_url=base_url,
    api_key=token_provider(),
)   

response = client.responses.create(
    model="gpt-4o-mini", # Replace with your deployment name, not the model ID 
    input="What are the top 3 benefits of cloud computing? Be concise.",
    max_output_tokens=500,
)

print(f"Response: {response.output_text}")
print(f"Status:   {response.status}")
print(f"Output tokens: {response.usage.output_tokens}")

Response: 1. **Scalability**: Easily adjust resources up or down based on demand, allowing for flexible growth without significant upfront investment.

2. **Cost Efficiency**: Reduces hardware and maintenance costs by offering a pay-as-you-go model, lowering overall IT expenses.

3. **Accessibility**: Enables access to data and applications from anywhere with an internet connection, fostering collaboration and remote work.
Status:   completed
Output tokens: 80


### 1.2- Generar respuesta estructurada en formato JSON.
Generación de respuesta estructurada en JSON.

[Documentación](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/structured-outputs?tabs=python-secure%2Cdotnet-entra-id&pivots=programming-language-python)


In [10]:
from pydantic import BaseModel
from openai import OpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(), "https://ai.azure.com/.default"
)

client = OpenAI(  
  base_url = "https://pruebauno.services.ai.azure.com/api/projects/projuno/openai/v1",  
  api_key=token_provider(),
)

class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]
    equipo_Ganador: str

completion = client.beta.chat.completions.parse(
    model="gpt-4o-mini", # replace with the model deployment name of your gpt-4o 2024-08-06 deployment
    messages=[
        {"role": "system", "content": "Extract the event information."},
        {"role": "user", "content": "El Atleti va a ganar la Champions League y Mario que es del barça va a llorar."},
    ],
    response_format=CalendarEvent,
)

event = completion.choices[0].message.parsed

print(event)
print(completion.model_dump_json(indent=2))

name='Final de la Champions League' date='2023-06-10' participants=['Atletico de Madrid', 'FC Barcelona'] equipo_Ganador='Atletico de Madrid'
{
  "id": "chatcmpl-DT1GaCmfOKBaAFlIrqAmaw49mDph0",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "{\"name\":\"Final de la Champions League\",\"date\":\"2023-06-10\",\"participants\":[\"Atletico de Madrid\",\"FC Barcelona\"],\"equipo_Ganador\":\"Atletico de Madrid\"}",
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": null,
        "parsed": {
          "name": "Final de la Champions League",
          "date": "2023-06-10",
          "participants": [
            "Atletico de Madrid",
            "FC Barcelona"
          ],
          "equipo_Ganador": "Atletico de Madrid"
        }
      },
      "content_filter_results": {
        "hate": {
          "filt

c:\Users\Alumno_AI\Downloads\foundry\practicaFoundry\.venv\Lib\site-packages\pydantic\main.py:528: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=CalendarEvent(name='Final...or='Atletico de Madrid'), input_type=CalendarEvent])
  return self.__pydantic_serializer__.to_json(


### 1.3- Implementar y demostrar Guardrails. 
Crear Guardrails para el modelo, documentar el proceso y hacer pruebas contra el modelo.

[Documentación](https://learn.microsoft.com/en-us/azure/foundry/guardrails/how-to-create-guardrails?tabs=python)

### Entregable: 
Notebook con código que muestre la llamada al endpoint del modelo para cada caso, ejemplos de prompts, validación del JSON recibido y una sección que muestre cómo se configuran y activan los Guardrails.



In [14]:
#TODO

---

## 2) Reasoning y Function Calling
Objetivo: Practicar con modelos razonadores y ver el function calling

### 2.1- Razonamiento
Desplegar un modelo razonador y parametrizar distintos grados de razonamiento (low, medium, high)

[Documentación](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/reasoning?tabs=csharp%2Cgpt-5)

### 2.2- Function calling
Activar un motor de búsqueda web para probar llamadas a funciones (`function calling`) que recuperen información externa.

[Documentación](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/web-search)

⭐ Suma puntos usar deep research o hacer function calling con una función custom.


### Entregable: 
Notebook que muestre:
	- Ejemplos comparativos (misma tarea con distintos niveles de razonamiento).
	- Integración del web search y ejemplo de `function calling` que combine resultados externos con la respuesta del modelo.

---

## 3) Modelos Multimodales
### Objetivo: 
Desplegar un modelo multimodal y probar interacciones que involucren imágenes, audio y/o texto combinado (por ejemplo: describir una imagen, transcribir audio y responder preguntas sobre su contenido, etc.).

[Documentación](https://learn.microsoft.com/en-us/azure/foundry-classic/foundry-models/how-to/use-chat-multi-modal?context=%2Fazure%2Ffoundry%2Fcontext%2Fcontext&pivots=programming-language-python)

### Entregable: 
Notebook con llamadas al endpoint multimodal mostrando varios ejemplos: subida/consulta de imágenes, audios y prompts mixtos; incluir control de formatos y manejo de respuestas (texto y/o estructuras).

---

Formato y criterios de entrega
- Cada parte debe entregarse como un notebook `.ipynb` autocontenido que incluya:
	- Sección de configuración / credenciales (explicando cómo configurar variables de entorno localmente).
	- Código reproducible conecta a un modelo ya desplegado, realiza las llamadas y procesa las respuestas.
	- Celdas de explicación y resultados visibles (salidas, figuras, JSON validados).
	- Una sección final de conclusiones y problemas encontrados.